# SIC 2026 · Inteligencia Artificial
## Clase 11: Del scraping a las reseñas: preprocesamiento, TF-IDF y primeras aplicaciones de PLN

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Ayer terminamos el ejercicio de países y llegamos hasta el scraping, donde la página de Samsung nos ganó. Hoy partimos justo ahí: entendemos qué pasó, conseguimos **reseñas reales** de productos Samsung desde una API y con ellas recorremos el resto del flujo de PLN.

| Hora | Parte | Qué vamos a hacer |
|---|---|---|
| 18:00 a 18:25 | 1. Lo que pasó con Samsung | Páginas estáticas y dinámicas, y cómo saber cuál es cuál |
| 18:25 a 19:00 | 2. Datos desde una API | Descargar reseñas reales en JSON y explorarlas |
| 19:00 a 19:30 | 3. Expresiones regulares | Retomar el ejercicio de porcentajes y limpiar reseñas |
| 19:30 a 19:40 | Pausa | |
| 19:40 a 20:30 | 4. Preprocesamiento | Minúsculas, stopwords, stemming y lematización |
| 20:30 a 21:10 | 5. TF-IDF | Convertir textos en números |
| 21:10 a 21:55 | 6. Aplicaciones | ¿De qué hablan las reseñas? ¿Son positivas o negativas? |
| 21:55 a 22:00 | Cierre | |

Cada paso tiene la misma estructura: **qué vamos a hacer**, **el código** y **qué deberías ver**. Los ejercicios tienen verificación y las soluciones están al final.

**Antes de empezar:** *Archivo > Guardar una copia en Drive*.

---
# Paso 0. Preparar el entorno

**Qué vamos a hacer:** instalar el modelo de español de spaCy, cargar librerías y definir `revisar`. Tarda alrededor de un minuto.

In [ ]:
!python -m spacy download es_core_news_sm -q

import re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download("stopwords", quiet=True)

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    """Imprime si un ejercicio está bien resuelto."""
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

NAVEGADOR = {"User-Agent": "Mozilla/5.0 (curso SIC 2026)", "Accept-Language": "es-CL,es;q=0.9"}
print("Listo.")

---
# Parte 1. Lo que pasó con Samsung

Ayer pedimos con `requests` la página de búsqueda de Samsung. La respuesta fue `200` (todo bien) y trajo más de 300.000 caracteres de HTML, pero `soup.find(...)` devolvió `None`: **los productos no estaban en el HTML**. ¿Por qué?

## Paso 1.1. Páginas estáticas y dinámicas

| | Página estática | Página dinámica |
|---|---|---|
| Cómo se arma | El servidor envía el HTML completo | El servidor envía un "cascarón" y el navegador lo completa ejecutando JavaScript |
| Qué recibe `requests` | Todo el contenido | Solo el cascarón: `requests` no ejecuta JavaScript |
| Ejemplos | Wikipedia, blogs, muchos sitios de gobierno | Tiendas en línea, buscadores, redes sociales |

**Cómo saberlo sin adivinar:** busca en `res.text` algo que ves en la página del navegador (un nombre de producto, o la clase que encontraste con el inspector). Si no está, la página es dinámica.

**Qué vamos a hacer:** aplicar esa prueba a una página estática y a la de Samsung.

In [ ]:
def es_estatica(url, algo_que_deberia_aparecer):
    """Descarga una página y revisa si un texto o una clase aparece en el HTML que recibe requests."""
    try:
        res = requests.get(url, headers=NAVEGADOR, timeout=15)
        presente = algo_que_deberia_aparecer in res.text
        print(f"Estado {res.status_code} | {len(res.text):,} caracteres | ¿aparece '{algo_que_deberia_aparecer}'? {presente}")
        return presente
    except Exception as e:
        print("No se pudo descargar:", type(e).__name__)
        return None

print("Wikipedia:")
es_estatica("https://es.wikipedia.org/wiki/Procesamiento_de_lenguajes_naturales", "lenguaje natural")
print("\nBuscador de Samsung:")
es_estatica("https://www.samsung.com/cl/aisearch/?searchvalue=Lavadora%20de%2017%20kg", "pd21-product-card__name")

**Qué deberías ver:** en Wikipedia, `True`: el texto viene en el HTML. En Samsung, `False`: la clase de las tarjetas de producto no viene, porque se agrega después con JavaScript. (Si la red de Colab falla, verás un mensaje de error: la idea es la misma.)

## Paso 1.2. El otro error: `find` y `find_all`

Ayer había un segundo problema que habría aparecido igual:

- `find` devuelve **el primer** elemento que coincide (o `None` si no hay ninguno).
- `find_all` devuelve **una lista** con todos los que coinciden (vacía si no hay ninguno).

**Qué vamos a hacer:** probar ambos con un HTML pequeño escrito a mano, que sí trae los productos.

In [ ]:
html_ejemplo = """
<div class="tarjeta"><p class="nombre">Lavadora 17 kg</p><span class="precio">$399.990</span></div>
<div class="tarjeta"><p class="nombre">Lavadora 21 kg</p><span class="precio">$549.990</span></div>
<div class="tarjeta"><p class="nombre">Secadora 9 kg</p><span class="precio">$459.990</span></div>
"""
sopa = BeautifulSoup(html_ejemplo, "html.parser")

print("find:    ", sopa.find(class_="nombre"))
print("find_all:", sopa.find_all(class_="nombre"))
print("\nNombres limpios:", [n.get_text(strip=True) for n in sopa.find_all(class_="nombre")])

**Qué deberías ver:** `find` devuelve una sola etiqueta; `find_all`, una lista de tres. Con `get_text(strip=True)` se obtiene solo el texto.

### Ejercicio 1
Con `sopa` y `find_all`, crea `precios`: una **lista de números enteros** con los tres precios (por ejemplo `399990`). Pista: saca el texto y quita `$` y los puntos con `.replace()`.

In [ ]:
precios = None  # TU CÓDIGO AQUÍ
print(precios)

In [ ]:
revisar(lambda: precios == [399990, 549990, 459990])

## Paso 1.3. Tres caminos para una página dinámica

1. **Buscar la API que usa la página** (el mejor). La página dinámica pide sus datos a algún servidor, casi siempre en formato **JSON**. Esa petición se ve en el navegador: F12, pestaña *Network*, filtro *Fetch/XHR*. Si encontramos esa URL, la consultamos directo con `requests`: es rápido y liviano.
2. **Usar un navegador automatizado** (Selenium o Playwright). Abre un navegador real, espera que se ejecute el JavaScript y entrega el HTML completo. Funciona casi siempre, pero es lento y pesado.
3. **Leer datos estructurados de la página** (JSON-LD). Algunas páginas incluyen sus datos principales en un bloque de JSON dentro del HTML. Sirve para nombres y códigos, no siempre para precios.

El año pasado, una estudiante del curso siguió el **camino 1** y encontró la API que Samsung usa para mostrar las reseñas de sus productos. Eso es lo que haremos ahora.

**Buenas prácticas:** son APIs internas del sitio, así que las usamos con fines educativos, con pocas peticiones y una pausa entre ellas.

---
# Parte 2. Reseñas reales desde una API

## Paso 2.1. ¿Qué es una API y qué es JSON?

> **Definición. API:** una dirección a la que un programa le pide datos y recibe una respuesta ordenada, pensada para máquinas y no para personas.
>
> **Definición. JSON:** un formato de texto para datos, con llaves `{}` para objetos (como diccionarios de Python) y corchetes `[]` para listas. `res.json()` lo convierte directo en diccionarios y listas de Python.

Las reseñas de los productos de Samsung Chile las entrega **Bazaarvoice**, un servicio de reseñas. La API recibe el **SKU** del producto (su código) y devuelve hasta 100 reseñas por página.

## Paso 2.2. Descargar las reseñas

**Qué vamos a hacer:** pedir las reseñas de tres productos (relojes y pulseras Galaxy), juntar las páginas, quitar reseñas repetidas y quedarnos con la nota, el título y el texto.

**Si la API no responde** (Samsung puede cambiarla en cualquier momento), el notebook usa un respaldo de 60 reseñas **ficticias** escritas para esta clase, con el mismo formato. Todo lo que sigue funciona con cualquiera de las dos.

In [ ]:
URL_RESENAS = "https://apps.bazaarvoice.com/bfd/v1/clients/samsung-es_cl/api-products/cv2/resources/data/reviews.json"
HEADERS_API = {
    "Accept": "application/json",
    "X-Bazaarvoice-Api-Version": "5.5",
    "X-Bazaarvoice-Platform-Version": "2",
    "Bv-Bfd-Token": "17499,main_site,es_CL",
    "Origin": "https://www.samsung.com",
}
SKUS = ["SM-L705FZB1CHO", "SM-L320NZSALTA", "SM-R390NZAALTA"]

def pagina_de_resenas(sku, offset):
    """Pide una página de hasta 100 reseñas de un producto."""
    parametros = {
        "resource": "reviews", "action": "REVIEWS_N_STATS",
        "filter": [f"productid:eq:{sku}", "contentlocale:eq:es_CL,es*"],
        "apiversion": "5.5", "displaycode": "17499-es_cl",
        "sort": "submissiontime:desc", "limit": 100, "offset": offset,
    }
    res = requests.get(URL_RESENAS, params=parametros, headers=HEADERS_API, timeout=15)
    res.raise_for_status()
    return res.json().get("response", {}).get("Results", [])

def descargar_resenas(skus, max_paginas=2):
    todas = []
    for sku in skus:
        for pagina in range(max_paginas):
            resultados = pagina_de_resenas(sku, offset=pagina * 100)
            print(f"{sku} | página {pagina + 1}: {len(resultados)} reseñas")
            todas.extend(resultados)
            time.sleep(1)                       # pausa para no saturar la API
            if len(resultados) < 100:           # no hay más páginas
                break
    return pd.DataFrame(todas)

In [ ]:
RESPALDO = [
    (5, 'Excelente compañero para correr', 'Lo uso todos los días para correr y el GPS es muy preciso. La batería dura casi dos días con el monitoreo activado.'),
    (5, 'Me encanta el seguimiento del sueño', 'El análisis del sueño es muy completo y me ayudó a ordenar mis horarios. La pantalla se ve perfecta incluso al sol.'),
    (4, 'Muy buen reloj, batería justa', 'Cumple con todo lo que promete, las notificaciones llegan rápido. La batería dura un día y medio, esperaba un poco más.'),
    (5, 'Pantalla espectacular', 'La pantalla es brillante y nítida, se lee perfecto bajo el sol. Muy cómodo para usar todo el día.'),
    (1, 'La batería no dura nada', 'Con el GPS activado la batería no alcanza a durar un día completo. Muy decepcionado para el precio que tiene.'),
    (2, 'Problemas de conexión con el celular', 'Se desconecta del celular varias veces al día y pierdo las notificaciones. Tuve que reinstalar la aplicación dos veces.'),
    (5, 'Ideal para el gimnasio', 'Registra bien los entrenamientos y el ritmo cardíaco. La aplicación muestra todo muy claro.'),
    (4, 'Buen producto, despacho rápido', 'Llegó en dos días y bien embalado. El reloj funciona muy bien, aunque la correa es un poco dura al principio.'),
    (1, 'Dejó de cargar al mes', 'Al mes de uso dejó de cargar. El servicio técnico se demoró tres semanas en responder. No lo recomiendo.'),
    (5, 'El mejor reloj que he tenido', 'Diseño elegante, pantalla grande y la batería dura bastante. Las funciones de salud son muy útiles.'),
    (3, 'Cumple, pero es caro', 'Funciona bien, pero por el precio esperaba más autonomía de batería. La pantalla es muy buena.'),
    (2, 'La aplicación es lenta', 'La aplicación del celular tarda mucho en sincronizar los datos del sueño. A veces no sincroniza nada.'),
    (5, 'Perfecto para monitorear la salud', 'Mide el ritmo cardíaco, el oxígeno y el estrés. Me da tranquilidad poder revisar mi salud cada día.'),
    (4, 'Muy cómodo', 'Casi no se siente en la muñeca y se puede dormir con él. La batería podría durar más.'),
    (1, 'Pantalla rayada en una semana', 'La pantalla se rayó a la semana de uso normal. Esperaba un vidrio más resistente.'),
    (5, 'Excelente calidad precio', 'Lo compré en oferta y es una maravilla. Batería, pantalla y aplicación funcionan muy bien.'),
    (2, 'El GPS falla', 'El GPS pierde la señal en la ciudad y los recorridos quedan cortados. Para correr no me sirve.'),
    (5, 'Notificaciones perfectas', 'Recibo los mensajes y llamadas sin sacar el celular. La conexión con el teléfono es estable.'),
    (4, 'Buen seguimiento del sueño', 'El seguimiento del sueño es bastante preciso. Me gustaría que la aplicación diera más consejos.'),
    (1, 'No reconoce el celular', 'Después de la actualización ya no se conecta con mi celular. El soporte no me dio solución.'),
    (5, 'Muy buena batería', 'Con uso normal la batería me dura casi dos días completos. Carga rápido.'),
    (3, 'Correa incómoda', 'El reloj es bueno pero la correa que trae es incómoda y me irrita la piel. Tuve que comprar otra.'),
    (5, 'Lo recomiendo', 'Funciona perfecto, la pantalla es hermosa y el seguimiento del ejercicio es muy completo.'),
    (2, 'Mide mal el ritmo cardíaco', 'El ritmo cardíaco marca valores muy distintos a los de mi otro dispositivo. No confío en la medición.'),
    (4, 'Bonito y funcional', 'Se ve muy bien y cumple. La batería es lo único que podría mejorar.'),
    (5, 'Gran regalo', 'Se lo regalé a mi papá y lo usa todos los días para caminar. La pantalla es grande y fácil de leer.'),
    (1, 'Muy caro para lo que ofrece', 'Por ese precio esperaba una batería mucho mejor. Tengo que cargarlo todas las noches.'),
    (5, 'Excelente para nadar', 'Lo uso en la piscina sin problemas y registra bien las vueltas. Resistente al agua de verdad.'),
    (4, 'Buena aplicación', 'La aplicación es completa y fácil de usar. A veces la sincronización tarda un poco.'),
    (2, 'Se reinicia solo', 'Se reinicia solo varias veces a la semana. Lo envié al servicio técnico y sigue igual.'),
    (5, 'Pantalla siempre encendida', 'La pantalla siempre encendida es muy útil y la batería igual aguanta todo el día.'),
    (3, 'Bueno, sin más', 'Hace lo básico bien. Nada que me sorprenda, pero tampoco me ha fallado.'),
    (5, 'Mide muy bien el sueño', 'Me sorprendió lo bien que mide las fases del sueño. Ahora duermo mejor.'),
    (1, 'Despacho tardío', 'El despacho se demoró tres semanas y nadie respondía los correos. El producto está bien, la atención pésima.'),
    (4, 'Liviano y rápido', 'El reloj es liviano y la interfaz es muy rápida. La batería es justa si se usa el GPS.'),
    (5, 'Funciones de salud increíbles', 'El electrocardiograma y la presión son funciones muy útiles. Excelente para cuidar la salud.'),
    (2, 'Batería se agota rápido', 'Desde la última actualización la batería se agota en medio día. Antes duraba mucho más.'),
    (5, 'Muy satisfecha', 'Pantalla nítida, buena batería y se conecta sin problemas con mi celular.'),
    (4, 'Recomendable para correr', 'El GPS es preciso y el ritmo cardíaco es confiable. La correa podría ser más cómoda.'),
    (1, 'Llegó defectuoso', 'Llegó con la pantalla con un pixel muerto. El cambio tomó más de un mes.'),
    (5, 'Super cómodo para dormir', 'No molesta para dormir y el análisis del sueño es muy bueno. La batería aguanta bien.'),
    (3, 'Pantalla buena, batería regular', 'La pantalla es excelente pero la batería no me dura el día completo con todo activado.'),
    (5, 'Me ayudó a moverme más', 'Los recordatorios para moverme y el seguimiento de pasos me motivaron a hacer más ejercicio.'),
    (2, 'La conexión bluetooth falla', 'Se pierde la conexión bluetooth con el celular constantemente. Muy frustrante.'),
    (4, 'Muy buen diseño', 'El diseño es elegante y la pantalla se ve muy bien. La aplicación podría ser más intuitiva.'),
    (5, 'Excelente para entrenar', 'Registra todo tipo de ejercicio y el ritmo cardíaco es exacto. Muy contento con la compra.'),
    (1, 'Servicio técnico pésimo', 'El reloj falló y el servicio técnico no me dio solución en dos meses. No vuelvo a comprar.'),
    (5, 'Batería de dos días', 'La batería dura dos días completos con uso normal. Muy contento con la autonomía.'),
    (4, 'Sincroniza bien', 'Se sincroniza bien con la aplicación del celular y las notificaciones llegan al instante.'),
    (2, 'Correa se rompió', 'La correa se rompió al segundo mes. Para un producto de este precio es inaceptable.'),
    (5, 'Pantalla y salud excelentes', 'La pantalla es preciosa y las funciones de salud son muy completas. Lo uso para controlar mi presión.'),
    (3, 'Precio alto', 'Es un buen reloj, pero el precio es demasiado alto comparado con otros modelos.'),
    (5, 'Perfecto', 'Todo funciona perfecto: batería, pantalla, sueño y ejercicio. Muy recomendable.'),
    (1, 'No sincroniza el sueño', 'Los datos del sueño nunca aparecen en la aplicación. Probé de todo y no funciona.'),
    (4, 'Muy útil en el día a día', 'Uso las notificaciones y el seguimiento de pasos todo el día. La batería alcanza justo.'),
    (5, 'Calidad premium', 'Materiales de muy buena calidad y una pantalla excelente. Vale lo que cuesta.'),
    (2, 'Mediciones poco confiables', 'Los pasos y las calorías no coinciden con la realidad. No confío en los datos.'),
    (5, 'Buenísimo para caminar', 'Lo uso para caminar todos los días. El GPS y los pasos son muy precisos.'),
    (4, 'Cumple lo prometido', 'Funciona bien y la batería dura lo que dicen. La aplicación tiene muchas opciones.'),
    (1, 'Pantalla se apaga sola', 'La pantalla se apaga sola y hay que reiniciarlo. Muy mala experiencia.'),
]

try:
    crudo = descargar_resenas(SKUS)
    resenas = (crudo.drop_duplicates(subset="Id")
                    .rename(columns={"Rating": "nota", "Title": "titulo", "ReviewText": "texto"})
                    [["nota", "titulo", "texto"]])
    if len(resenas) < 30:
        raise ValueError("Muy pocas reseñas")
    FUENTE = "API de Samsung (Bazaarvoice)"
except Exception as e:
    print("No se pudo usar la API:", type(e).__name__, "| Uso las reseñas de respaldo (ficticias).")
    resenas = pd.DataFrame(RESPALDO, columns=["nota", "titulo", "texto"])
    FUENTE = "respaldo ficticio"

resenas = resenas.dropna(subset=["texto"]).reset_index(drop=True)
resenas["titulo"] = resenas["titulo"].fillna("")
print(f"\nFuente: {FUENTE} | Reseñas: {len(resenas)}")
resenas.head()

**Qué deberías ver:** si la API funciona, varias páginas descargadas y algunos cientos de reseñas reales. Si no, el mensaje de respaldo y 60 reseñas.

**Por qué `drop_duplicates(subset="Id")`:** Bazaarvoice agrupa las reseñas por familia de producto, así que el mismo reloj en distintos colores puede devolver las mismas reseñas. Cada reseña tiene un `Id` único, y así quitamos las repetidas.

## Paso 2.3. Explorar

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
resenas["nota"].value_counts().sort_index().plot(kind="bar", ax=ejes[0], color="#1428A0")
ejes[0].set_title("Reseñas por nota"); ejes[0].set_xlabel("Nota (1 a 5)")
resenas["texto"].str.split().str.len().plot(kind="hist", bins=20, ax=ejes[1], color="#FF6B35")
ejes[1].set_title("Largo de las reseñas"); ejes[1].set_xlabel("Palabras")
plt.tight_layout(); plt.show()

print("Ejemplo de reseña con nota 1:\n", resenas.loc[resenas["nota"] == 1, "texto"].iloc[0])

**Qué deberías ver:** casi siempre hay muchas más reseñas con nota 5 que con nota 1. Es un **desbalance**, igual que en el caso de fuga de clientes de la Clase 6, y lo tendremos en cuenta en la Parte 6.

### Ejercicio 2
Guarda en `prop_positivas` la **proporción** de reseñas con nota 4 o 5 (un número entre 0 y 1).

In [ ]:
prop_positivas = None  # TU CÓDIGO AQUÍ
print(prop_positivas)

In [ ]:
revisar(lambda: np.isclose(prop_positivas, (resenas["nota"] >= 4).mean()))

---
# Parte 3. Expresiones regulares

Repaso rápido de la tabla de ayer:

| Símbolo | Significa | Ejemplo |
|---|---|---|
| `\d` | Un dígito | `\d{4}` encuentra `2026` |
| `\w` | Letra, dígito o `_` | `\w+` encuentra una palabra |
| `\s` | Un espacio | |
| `+` / `*` / `?` | Una o más / cero o más / opcional | |
| `[...]` | Uno de estos caracteres | `[kK]` |
| `(?:...)` | Agrupa sin guardar aparte | `(?:\.\d{3})*` |
| `\` | Usa el símbolo literal | `\.` es un punto |

`re.findall(patrón, texto)` extrae y `re.sub(patrón, reemplazo, texto)` reemplaza.

### Ejercicio 3 (el que quedó pendiente ayer)
Escribe `patron_porcentaje` para encontrar `45%`, `3,5%` y `100%`. Pista: dígitos, opcionalmente una coma con más dígitos, y `%`.

In [ ]:
texto_porcentajes = "La aprobación subió a 45% y la inflación anual fue de 3,5%. La meta es llegar al 100% de cobertura."

patron_porcentaje = None  # TU CÓDIGO AQUÍ, por ejemplo r"..."
print(re.findall(patron_porcentaje, texto_porcentajes) if patron_porcentaje else None)

In [ ]:
revisar(lambda: re.findall(patron_porcentaje, texto_porcentajes) == ["45%", "3,5%", "100%"])

## Paso 3.1. Limpiar texto de usuarios

Las reseñas, los comentarios y las publicaciones traen ruido típico: enlaces, menciones, signos repetidos y espacios de más.

**Qué vamos a hacer:** escribir una función `limpiar` con varias reglas de `re.sub` y probarla.

In [ ]:
def limpiar(texto):
    texto = re.sub(r"https?://\S+", " ", texto)        # enlaces: http o https, y todo hasta el siguiente espacio
    texto = re.sub(r"@\w+", " ", texto)                # menciones como @samsungchile
    texto = re.sub(r"([!?.])\1+", r"\1", texto)         # signos repetidos: "!!!" pasa a "!"
    texto = re.sub(r"\s+", " ", texto)                  # varios espacios o saltos de línea pasan a uno
    return texto.strip()

ejemplo_sucio = "Llegó rápido!!!   Lo compré en https://www.samsung.com/cl y @samsungchile respondió altiro...\n\nMuy bueno"
print("Antes:  ", repr(ejemplo_sucio))
print("Después:", repr(limpiar(ejemplo_sucio)))

**Qué deberías ver:** sin el enlace, sin la mención, con un solo signo de exclamación y sin espacios de más.

**Cómo leer `([!?.])\1+`:** los paréntesis **guardan** el signo encontrado, y `\1` significa "el mismo signo que guardé". Entonces encuentra un signo seguido de una o más repeticiones de sí mismo.

### Ejercicio 4
Escribe `patron_hashtag` para extraer los hashtags (como `#GalaxyWatch`) del texto de la celda.

In [ ]:
texto_hashtags = "Feliz con mi nuevo reloj #GalaxyWatch #Salud. Lo uso para correr todos los días #Running"

patron_hashtag = None  # TU CÓDIGO AQUÍ
print(re.findall(patron_hashtag, texto_hashtags) if patron_hashtag else None)

In [ ]:
revisar(lambda: re.findall(patron_hashtag, texto_hashtags) == ["#GalaxyWatch", "#Salud", "#Running"])

Aplicamos la limpieza a todas las reseñas. Juntamos el título y el texto, porque el título suele resumir la opinión.

In [ ]:
resenas["completo"] = (resenas["titulo"] + ". " + resenas["texto"]).apply(limpiar)
resenas[["nota", "completo"]].head()

---
# Parte 4. Preprocesamiento

Lo hacemos paso a paso con una sola frase, para ver qué hace cada paso.

In [ ]:
frase = "Los Bancos Centrales subieron las tasas de interés, pero la economía chilena creció 2,5% en 2026."

paso1 = frase.lower()                                   # 1. minúsculas
paso2 = re.sub(r"[^a-záéíóúñü\s]", " ", paso1)          # 2. todo lo que no sea letra pasa a espacio
tokens = paso2.split()                                  # 3. tokenizar

print("1. Minúsculas: ", paso1)
print("2. Solo letras:", paso2)
print("3. Tokens:     ", tokens, f"({len(tokens)} tokens)")

**Qué deberías ver:** la frase en minúsculas, luego sin comas ni números, y luego como una lista de 14 tokens.

`[^...]` significa "cualquier carácter que **no** esté en la lista". Ojo: aquí quitamos los números porque para hablar de temas no aportan; en otros problemas (reclamos con montos, por ejemplo) sí importan.

## Paso 4.1. Stopwords, y una decisión importante

> **Definición. Stopwords:** palabras muy frecuentes que casi no aportan significado sobre el tema, como "el", "de", "que" o "pero". NLTK trae 313 en español.

In [ ]:
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("spanish"))
print("Sin stopwords:", [t for t in tokens if t not in STOPWORDS])
print("\n¿'no' es stopword?", "no" in STOPWORDS, "| ¿'muy'?", "muy" in STOPWORDS, "| ¿'sin'?", "sin" in STOPWORDS)

**Qué deberías ver:** desaparecen "los", "las", "de", "pero", "la" y "en". Y **"no", "muy" y "sin" están en la lista de stopwords**.

Eso es un problema para las reseñas: "funciona" y "**no** funciona" quedarían iguales. **La lista de stopwords depende del problema.** Para hablar de temas, "no" sobra; para saber si una opinión es positiva o negativa, es de las palabras más importantes. Por eso creamos una segunda lista que conserva las negaciones e intensificadores.

In [ ]:
CONSERVAR = {"no", "sin", "nada", "nunca", "muy", "más", "poco"}
STOPWORDS_OPINION = STOPWORDS - CONSERVAR
print("Palabras que conservamos:", sorted(CONSERVAR))

## Paso 4.2. Stemming y lematización

| | Stemming | Lematización |
|---|---|---|
| Qué hace | Corta la palabra para dejar su raíz | Busca la forma de diccionario |
| "subieron" | `sub` | `subir` |
| "chilena" | `chilen` | `chileno` |
| Ventaja | Muy rápido | Siempre es una palabra real |
| Herramienta | NLTK `SnowballStemmer` | spaCy |

In [ ]:
from nltk.stem import SnowballStemmer
import spacy

stemmer = SnowballStemmer("spanish")
nlp = spacy.load("es_core_news_sm")

palabras = [t for t in tokens if t not in STOPWORDS]
pd.DataFrame({"palabra": palabras,
              "stemming": [stemmer.stem(t) for t in palabras],
              "lematización": [t.lemma_ for t in nlp(" ".join(palabras))]})

**Qué deberías ver:** el stemming deja raíces como `econom`; la lematización, palabras completas como `economía`.

## Paso 4.3. Todo junto

**Qué vamos a hacer:** una función `preprocesar` con dos opciones: el método de normalización y la lista de stopwords.

In [ ]:
def preprocesar(texto, metodo="stemming", stopwords_usar=STOPWORDS):
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto.lower())        # minúsculas y solo letras
    tokens = [t for t in texto.split()
              if t not in stopwords_usar and (len(t) > 2 or t == "no")]   # sin stopwords ni palabras muy cortas (salvo "no")
    if metodo == "stemming":
        tokens = [stemmer.stem(t) for t in tokens]
    elif metodo == "lematizacion":
        tokens = [t.lemma_ for t in nlp(" ".join(tokens))]
    return " ".join(tokens)

opinion = "No funciona bien, la batería no dura nada y es muy cara."
print("Para temas:   ", preprocesar(opinion))
print("Para opinión: ", preprocesar(opinion, stopwords_usar=STOPWORDS_OPINION))

**Qué deberías ver:** con la lista normal, la opinión pierde "no", "nada" y "muy", y parece neutral. Con la lista para opiniones, se conserva lo negativo.

### Ejercicio 5
Usa `preprocesar` (con stemming y la lista normal) sobre `otra_frase` y guarda el resultado en `frase_procesada`.

In [ ]:
otra_frase = "Las jugadoras chilenas ganaron el campeonato y celebraron con la hinchada en el estadio."

frase_procesada = None  # TU CÓDIGO AQUÍ
print(frase_procesada)

In [ ]:
revisar(lambda: frase_procesada == preprocesar(otra_frase))

Preprocesamos las reseñas de las dos formas, porque las usaremos para dos preguntas distintas en la Parte 6.

In [ ]:
resenas["para_temas"] = resenas["completo"].apply(preprocesar)
resenas["para_opinion"] = resenas["completo"].apply(lambda t: preprocesar(t, stopwords_usar=STOPWORDS_OPINION))
resenas[["completo", "para_temas", "para_opinion"]].head(3)

---
# Parte 5. Del texto a números: TF-IDF

> **Bolsa de palabras:** cada documento se representa como el conteo de cada palabra, sin importar el orden.
>
> **TF-IDF:** multiplica la frecuencia de la palabra en el documento (TF) por qué tan rara es en el corpus (IDF). Premia las palabras características de un documento.

scikit-learn calcula:

$$\text{IDF} = \ln\left(\frac{1 + n}{1 + df}\right) + 1$$

donde **n** es el número de documentos y **df** el número de documentos que contienen la palabra.

## Paso 5.1. A mano

Corpus de 3 documentos: "gol partido gol", "partido tenis", "tasa interés tasa".

- **gol** aparece en 1 documento: IDF = ln(4/2) + 1 = **1,693**. En el documento 1 aparece 2 veces: TF-IDF = 2 × 1,693 = **3,386**.
- **partido** aparece en 2 documentos: IDF = ln(4/3) + 1 = **1,288**. En el documento 1 aparece 1 vez: TF-IDF = **1,288**.

"Partido" pesa menos porque es menos característica.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

mini_corpus = ["gol partido gol", "partido tenis", "tasa interés tasa"]
indices = ["doc 1", "doc 2", "doc 3"]

conteo = CountVectorizer()
print("Bolsa de palabras:")
display(pd.DataFrame(conteo.fit_transform(mini_corpus).toarray(), columns=conteo.get_feature_names_out(), index=indices))

sin_norma = TfidfVectorizer(norm=None)
print("TF-IDF sin normalizar (compáralo con el cálculo a mano):")
display(pd.DataFrame(sin_norma.fit_transform(mini_corpus).toarray().round(3), columns=sin_norma.get_feature_names_out(), index=indices))

tfidf = TfidfVectorizer()
print("TF-IDF normalizado (cada fila tiene largo 1):")
display(pd.DataFrame(tfidf.fit_transform(mini_corpus).toarray().round(3), columns=tfidf.get_feature_names_out(), index=indices))

**Qué deberías ver:** en la segunda tabla, `gol = 3,386` y `partido = 1,288` en el documento 1, igual que a mano. La tercera tabla divide cada fila por su largo, para que los documentos largos no tengan ventaja.

### Ejercicio 6
Con un cuarto documento, `"partido gol tenis"`, hay n = 4 documentos y "partido" aparece en 3. Calcula **a mano** su IDF con `np.log` y guárdalo en `idf_partido`.

In [ ]:
idf_partido = None  # TU CÓDIGO AQUÍ
print(idf_partido)

In [ ]:
_v = TfidfVectorizer().fit(mini_corpus + ["partido gol tenis"])
revisar(lambda: np.isclose(idf_partido, _v.idf_[list(_v.get_feature_names_out()).index("partido")]))

---
# Parte 6. Aplicaciones con las reseñas

Con las reseñas convertidas en números, respondemos dos preguntas que le importan a cualquier empresa:

1. **¿De qué hablan los clientes?** Agrupamiento con K-medias (no supervisado, Clases 7 a 10).
2. **¿La reseña es positiva o negativa?** Clasificación con regresión logística (supervisado, Clases 4 a 6).

## Paso 6.1. ¿De qué hablan? Agrupar por temas

**Qué vamos a hacer:** TF-IDF sobre el texto preprocesado "para temas", K-medias con k = 5, y para cada grupo: sus palabras clave, cuántas reseñas tiene y su nota promedio.

In [ ]:
from sklearn.cluster import KMeans

vect_temas = TfidfVectorizer(min_df=2)          # min_df=2: ignorar palabras que aparecen en una sola reseña
X_temas = vect_temas.fit_transform(resenas["para_temas"])
print("Matriz:", X_temas.shape, "(reseñas × palabras)")

K = 5
km = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X_temas)
resenas["tema"] = km.labels_

terminos = vect_temas.get_feature_names_out()
resumen_temas = pd.DataFrame({
    "palabras_clave": [", ".join(terminos[np.argsort(c)[::-1][:6]]) for c in km.cluster_centers_],
    "reseñas": np.bincount(km.labels_, minlength=K),
    "nota_promedio": resenas.groupby("tema")["nota"].mean().round(2).reindex(range(K)).values,
})
resumen_temas.sort_values("nota_promedio")

**Qué deberías ver:** grupos con palabras clave que se pueden nombrar, como batería, pantalla, conexión con el celular, sueño o salud. La nota promedio muestra **qué temas generan más molestia**: es exactamente lo que un equipo de producto quiere saber.

Leamos dos reseñas del tema con peor nota, para confirmar que el nombre tiene sentido:

In [ ]:
tema_peor = resenas.groupby("tema")["nota"].mean().idxmin()
for t in resenas.loc[resenas["tema"] == tema_peor, "completo"].head(2):
    print("-", t)

### Ejercicio 7
Guarda en `tema_mejor` el número del tema con la **nota promedio más alta**, y en `nombre_tema_mejor` un nombre corto que tú le pongas mirando sus palabras clave (este no se verifica).

In [ ]:
tema_mejor = None         # TU CÓDIGO AQUÍ
nombre_tema_mejor = ""    # TU NOMBRE AQUÍ
print(tema_mejor, nombre_tema_mejor)

In [ ]:
revisar(lambda: tema_mejor == resenas.groupby("tema")["nota"].mean().idxmax())

## Paso 6.2. ¿Positiva o negativa? Clasificar opiniones

**Qué vamos a hacer:**

1. Crear la variable objetivo: **positiva** (nota 4 o 5) o **negativa** (nota 1 o 2). Dejamos fuera las de nota 3, que son ambiguas.
2. Usar el texto preprocesado **para opinión** (el que conserva "no").
3. Armar un `Pipeline` con TF-IDF y regresión logística con `class_weight="balanced"`, por el desbalance.
4. Evaluar con validación cruzada y mirar **precision, recall y F1** de la clase negativa, que es la que más le importa a la empresa (Clase 7).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report

opiniones = resenas[resenas["nota"] != 3].copy()
opiniones["sentimiento"] = np.where(opiniones["nota"] >= 4, "positiva", "negativa")
print(opiniones["sentimiento"].value_counts())

clasificador = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
prediccion = cross_val_predict(clasificador, opiniones["para_opinion"], opiniones["sentimiento"], cv=cv)
print(classification_report(opiniones["sentimiento"], prediccion, digits=3))

**Qué deberías ver:** un reporte con precision, recall y F1 para cada clase. Normalmente la clase positiva tiene mejores números porque hay más ejemplos. **Miren el recall de la clase negativa:** de las reseñas negativas reales, ¿cuántas detecta el modelo?

`cross_val_predict` entrega, para cada reseña, la predicción de un modelo que **no la vio** al entrenar. Así el reporte es honesto.

### ¿Qué palabras aprendió el modelo?

In [ ]:
clasificador.fit(opiniones["para_opinion"], opiniones["sentimiento"])
coef = pd.Series(clasificador.named_steps["modelo"].coef_[0], index=clasificador.named_steps["tfidf"].get_feature_names_out())
# La clase "positiva" es la segunda en orden alfabético: coeficiente positivo empuja hacia "positiva"
pd.concat([coef.nsmallest(8), coef.nlargest(8)]).plot(kind="barh", figsize=(7, 5), color=["#D9534F"] * 8 + ["#1E9E73"] * 8,
                                                      title="Palabras que empujan hacia negativa (rojo) o positiva (verde)")
plt.show()

**Qué deberías ver:** **"no"** entre las palabras que más empujan hacia negativa. Si hubiéramos usado la lista normal de stopwords, esa palabra no existiría para el modelo: la decisión del Paso 4.1 importa.

Probemos con opiniones nuevas:

In [ ]:
nuevas = ["La batería dura dos días y la pantalla es excelente",
          "No sincroniza con el celular y el servicio técnico no responde",
          "Muy cómodo, lo uso para dormir y medir el sueño"]
for texto, pred in zip(nuevas, clasificador.predict([preprocesar(limpiar(t), stopwords_usar=STOPWORDS_OPINION) for t in nuevas])):
    print(f"{pred:9s} | {texto}")

### Ejercicio 8
Del reporte de clasificación, guarda en `f1_negativa` el **F1 de la clase negativa** (un número entre 0 y 1). Pista: `f1_score(..., pos_label="negativa")` con `opiniones["sentimiento"]` y `prediccion`.

In [ ]:
from sklearn.metrics import f1_score

f1_negativa = None  # TU CÓDIGO AQUÍ
print(f1_negativa)

In [ ]:
revisar(lambda: np.isclose(f1_negativa, f1_score(opiniones["sentimiento"], prediccion, pos_label="negativa")))

---
# Cierre

| Tema | Lo esencial |
|---|---|
| Scraping | Si lo que ves en el navegador no está en `res.text`, la página es dinámica. El mejor camino es encontrar la API en la pestaña *Network*. |
| BeautifulSoup | `find` devuelve uno (o `None`); `find_all`, una lista. |
| APIs y JSON | `res.json()` entrega diccionarios y listas de Python. Pocas peticiones, con pausa. |
| Regex | `re.sub` para limpiar enlaces, menciones y signos repetidos. |
| Stopwords | La lista depende del problema: para opiniones, "no" es clave. |
| TF-IDF | Premia lo característico de cada documento. |
| Aplicaciones | Temas con K-medias y sentimiento con regresión logística, evaluada con F1 de la clase que importa. |

**Para el capstone:** si su proyecto usa texto, este notebook es una plantilla completa: obtener, limpiar, preprocesar, TF-IDF, agrupar y clasificar.

Guarden el notebook en `ejercicios_estudiantes` como `clase11_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1

In [ ]:
precios = [int(p.get_text(strip=True).replace("$", "").replace(".", "")) for p in sopa.find_all(class_="precio")]
print(precios)

### Ejercicio 2

In [ ]:
prop_positivas = (resenas["nota"] >= 4).mean()
print(round(prop_positivas, 3))

### Ejercicio 3

In [ ]:
patron_porcentaje = r"\d+(?:,\d+)?%"
print(re.findall(patron_porcentaje, texto_porcentajes))

### Ejercicio 4
Un `#` seguido de una o más letras, dígitos o guiones bajos.

In [ ]:
patron_hashtag = r"#\w+"
print(re.findall(patron_hashtag, texto_hashtags))

### Ejercicio 5

In [ ]:
frase_procesada = preprocesar(otra_frase)
print(frase_procesada)

### Ejercicio 6

In [ ]:
idf_partido = np.log((1 + 4) / (1 + 3)) + 1
print(round(idf_partido, 3))

### Ejercicio 7

In [ ]:
tema_mejor = resenas.groupby("tema")["nota"].mean().idxmax()
print(tema_mejor, "|", resumen_temas.loc[tema_mejor, "palabras_clave"])

### Ejercicio 8

In [ ]:
f1_negativa = f1_score(opiniones["sentimiento"], prediccion, pos_label="negativa")
print(round(f1_negativa, 3))